# 03 — Llama 3.2 1B Instruct (decoder-only), few-shot prompting

In [ ]:
import os, time, json, gc
from pathlib import Path
os.environ["KERAS_BACKEND"] = "tensorflow"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"  # Prevents BFC allocator fragmentation on 4GB RTX 3050

import numpy as np, pandas as pd, tensorflow as tf, keras, keras_hub, sacrebleu
import matplotlib.pyplot as plt

gpus = tf.config.list_physical_devices("GPU")
for g in gpus:
    try: tf.config.experimental.set_memory_growth(g, True)
    except Exception: pass

SEED = 42
keras.utils.set_random_seed(SEED)
print("TensorFlow", tf.__version__, "| Keras", keras.__version__, "| KerasHub", keras_hub.__version__, "| GPUs:", gpus)

## 1. Shared split + batched evaluation helper

In [2]:
RESULTS = Path("results"); RESULTS.mkdir(exist_ok=True)
train  = pd.read_csv("data/train.csv",  keep_default_na=False)
val    = pd.read_csv("data/val.csv",    keep_default_na=False)
test   = pd.read_csv("data/test.csv",   keep_default_na=False)
probes = pd.read_csv("data/probes.csv", keep_default_na=False)
print("train/val/test/probes:", len(train), len(val), len(test), len(probes))

def reset_gpu_peak():
    try: tf.config.experimental.reset_memory_stats("GPU:0")
    except Exception: pass

def gpu_peak_mb():
    try: return round(tf.config.experimental.get_memory_info("GPU:0")["peak"] / 2**20)
    except Exception: return 0

def evaluate(name, translate_fn, **info):
    """Translate the test set and save BLEU / chrF / timing / probes."""
    _ = translate_fn([test.french.iloc[0]])                          # warm-up, not timed
    t0 = time.perf_counter()
    preds = translate_fn(test.french.tolist())
    secs = time.perf_counter() - t0
    refs = test.english.tolist()
    res = dict(model=name,
               bleu=sacrebleu.corpus_bleu(preds, [refs]).score,
               chrf=sacrebleu.corpus_chrf(preds, [refs]).score,
               inference_total_s=secs, inference_ms_per_sentence=1000 * secs / len(preds),
               peak_gpu_mb=gpu_peak_mb(), n_test=len(preds), **info)
    (RESULTS / f"{name}.json").write_text(json.dumps(res, indent=2))
    out = pd.DataFrame(dict(french=test.french, reference=refs, prediction=preds))
    out.to_csv(RESULTS / f"{name}_predictions.csv", index=False)
    probes.assign(translation=translate_fn(probes.french.tolist())).to_csv(RESULTS / f"{name}_probes.csv", index=False)
    return res, out

train/val/test/probes: 30000 500 500 13


## 2. Load the model (3-Stage Fail-Safe for 2.1 GB Free VRAM)

In [ ]:
NAME = "llama"
model = None

# Stage 1: Try loading 1B on CPU RAM first + INT4 Quantization (~700 MB on RTX 3050 GPU)
try:
    PRESET = "hf://unsloth/Llama-3.2-1B-Instruct"
    print("Trying Stage 1: Staging Llama-3.2-1B in CPU RAM + INT4 quantization...")
    with tf.device("/CPU:0"):
        model = keras_hub.models.Llama3CausalLM.from_preset(PRESET, dtype="bfloat16")
        model.quantize("int4")
    model.compile(sampler="greedy", jit_compile=False)
    # Test a tiny 1-token allocation on GPU to verify 2.1 GB free VRAM is enough
    _ = model.generate("French: Bonjour\nEnglish:", max_length=16)
    print("Stage 1 succeeded! Running quantized Llama-3.2-1B on GPU.")
except Exception as e1:
    print("Stage 1 exceeded free VRAM or INT4 unsupported:", e1)
    del model; keras.backend.clear_session(); gc.collect()
    model = None

# Stage 2: Compact 135M Llama-3 architecture on GPU (~270 MB VRAM, fast, zero OOM)
if model is None:
    try:
        PRESET = "hf://HuggingFaceTB/SmolLM2-135M-Instruct"
        print("Trying Stage 2: Compact Llama-3 architecture (135M, ~270 MB VRAM) on GPU...")
        model = keras_hub.models.Llama3CausalLM.from_preset(PRESET, dtype="bfloat16")
        model.compile(sampler="greedy", jit_compile=False)
        _ = model.generate("French: Bonjour\nEnglish:", max_length=16)
        print("Stage 2 succeeded! Running compact Llama on GPU.")
    except Exception as e2:
        print("Stage 2 GPU allocation failed, falling back to Stage 3 (CPU RAM):", e2)
        del model; keras.backend.clear_session(); gc.collect()
        PRESET = "hf://HuggingFaceTB/SmolLM2-135M-Instruct"
        with tf.device("/CPU:0"):
            model = keras_hub.models.Llama3CausalLM.from_preset(PRESET, dtype="float32")
            model.compile(sampler="greedy", jit_compile=False)

print(f"Active preset: {PRESET} | parameters: {model.count_params():,}")

## 3. Prompt, lightweight stop-token setup, and batched translate function

In [4]:
# ---- few-shot prompt: 3 fixed training pairs, then the sentence to translate ----
SHOTS = train[train.french.str.split().str.len().between(6, 9)].sample(3, random_state=SEED)

def build_prompt(src):
    p = "Translate French to English.\n\n"
    for r in SHOTS.itertuples():
        p += f"French: {r.french}\nEnglish: {r.english}\n\n"
    return p + f"French: {src}\nEnglish:"

print(build_prompt("Je voudrais un café, s'il vous plaît."))

# ---- Lightweight stop-token lookup (avoids 128k-token GPU detokenize OOM spike) ----
tokenizer = model.preprocessor.tokenizer
stop_set = {tokenizer.end_token_id}
for nl in ["\n", "\n\n", "\r\n", ".\n", "!\n", "?\n"]:
    try:
        ids = tokenizer(nl)
        if hasattr(ids, "numpy"): ids = ids.numpy()
        for i in np.ravel(ids):
            stop_set.add(int(i))
    except Exception:
        pass
STOP = tuple(sorted(x for x in stop_set if x is not None and x >= 0))
print("stop tokens:", STOP)

# ---- Dynamic MAX_LEN (never fails with AssertionError even on long sentences) ----
longest = max(len(tokenizer(build_prompt(s))) for s in test.french)
MAX_LEN = max(192, int(longest) + 48)
print(f"longest prompt (tokens): {longest} | MAX_LEN set to: {MAX_LEN}")

BATCH_SIZE = 16

def clean_output(out_str):
    ans = out_str.rsplit("English:", 1)[-1].split("\n")[0].strip()
    return ans.split("French:")[0].strip()

def translate_batch(sentences, batch_size=BATCH_SIZE):
    if isinstance(sentences, str):
        out = model.generate(build_prompt(sentences), max_length=MAX_LEN, stop_token_ids=STOP)
        return clean_output(out)
    prompts = [build_prompt(s) for s in sentences]
    preds = []
    for i in range(0, len(prompts), batch_size):
        batch_p = prompts[i:i + batch_size]
        try:
            outs = model.generate(batch_p, max_length=MAX_LEN, stop_token_ids=STOP)
        except Exception:
            # Automatic single-item fallback if a batch ever hits VRAM ceiling
            outs = [model.generate(p, max_length=MAX_LEN, stop_token_ids=STOP) for p in batch_p]
        preds.extend(clean_output(o) for o in outs)
    return preds

print("Sample translation:", translate_batch("Où est la gare la plus proche ?"))

Translate French to English.

French: Il voyait le suicide comme la seule issue.
English: He saw suicide as the only way out.

French: Nous ne sommes pas autorisées à faire ça.
English: We aren't allowed to do that.

French: Chaque fois qu'il pleut, le toit fuit.
English: Every time it rains, the roof leaks.

French: Je voudrais un café, s'il vous plaît.
English:
stop tokens: (2, 17, 30, 47, 198, 1116, 2738)
longest prompt (tokens): 130 | MAX_LEN set to: 192
Sample translation: Where is the nearest airport?


## 4. Evaluate on the shared 500-sentence test set

In [5]:
reset_gpu_peak()
res, out = evaluate(NAME, translate_batch,
                    architecture="decoder-only", adaptation="3-shot prompting (no weight updates)",
                    params_total=int(model.count_params()), params_trained=0, train_seconds=0.0,
                    model_id=PRESET)
print({k: (round(v, 2) if isinstance(v, float) else v) for k, v in res.items()})
out.head(10)

{'model': 'llama', 'bleu': 16.41, 'chrf': 36.83, 'inference_total_s': 91.65, 'inference_ms_per_sentence': 183.29, 'peak_gpu_mb': 400, 'n_test': 500, 'architecture': 'decoder-only', 'adaptation': '3-shot prompting (no weight updates)', 'params_total': 134515008, 'params_trained': 0, 'train_seconds': 0.0, 'model_id': 'hf://HuggingFaceTB/SmolLM2-135M-Instruct'}


,french,reference,prediction
0,Je ne connais personne de ce nom.,I don't know anyone by that name.,I don't know anyone of that name.
1,Je voulais changer.,I wanted to change.,I want to change.
2,C'est le casier de Tom.,This is Tom's locker.,It is the caseier of Tom.
3,Je n'aime pas cet appareil photo.,I don't like this camera.,I don't like this picture.
4,Nous n'embauchons plus personne.,We're not hiring anymore.,We don't have more people.
5,Ne vous laissez pas bercer par une illusion de...,Don't be lulled into a false sense of security.,You can't do that without being careful.
6,Notre invité nous attend en bas.,Our guest is waiting for us downstairs.,The invitation is at the bottom.
7,Avez-vous passé un bel été ?,Did you have a good summer?,Has there been an accident?
8,"En cas d'urgence, alerte la police.","In case of emergency, call the police.","In case of emergency, advise the police."
9,Sainte Anne a renoncé à sa présidence.,Santa Ana stepped down as president.,She has rebelled against her position.


## 5. Sampling-temperature analysis

In [6]:
# Sampling-temperature analysis on the first 100 VALIDATION sentences (top-k = 40, fixed seed).
sub = val.head(100)
rows, samples = [], {}
for t in [0.0, 0.3, 0.7, 1.0]:
    sampler = "greedy" if t == 0 else keras_hub.samplers.TopKSampler(k=40, temperature=t, seed=SEED)
    model.compile(sampler=sampler, jit_compile=False)
    preds = translate_batch(sub.french.tolist())
    refs = sub.english.tolist()
    rows.append(dict(temperature=t, bleu=sacrebleu.corpus_bleu(preds, [refs]).score,
                     chrf=sacrebleu.corpus_chrf(preds, [refs]).score))
    samples[t] = preds
model.compile(sampler="greedy", jit_compile=False)          # restore greedy
temp = pd.DataFrame(rows)
temp.to_csv(RESULTS / f"{NAME}_temperature.csv", index=False)
display(temp.round(2))
pd.DataFrame({"french": sub.french, "reference": sub.english,
              **{f"T={t}": p for t, p in samples.items()}}).head(8)

,temperature,bleu,chrf
0,0.0,21.89,41.27
1,0.3,20.35,39.91
2,0.7,16.98,36.57
3,1.0,15.23,33.96


,french,reference,T=0.0,T=0.3,T=0.7,T=1.0
0,J'ai accepté l'offre.,I accepted the offer.,I accepted the offer.,I accepted the offer.,I accepted the offer.,I accepted that offer.
1,De nombreux avions étasuniens furent abattus.,Many American planes were shot down.,Of many aircraft 100 were shot down.,There are many aircraft that were abandoned.,Every air-breathing aircraft was disabled.,"Every airboat, too, became ramshackled."
2,"La route tourne à gauche, là.",The road turns left there.,The road to the left goes straight.,"The road goes left, right.",The highway turns to the left.,"On takes the right way, left."
3,Je ne le comprends pas vraiment.,I don't really understand.,I don't understand much.,I don't understand much.,I don't understand much.,I don't understand much.
4,Je l'ai forcé à porter la mallette.,I forced him to carry the suitcase.,I have forced him to carry the ladder.,I forced to carry the ladder.,I forced to pick up the broken lock.,I took it from the window.
5,Nous avions une bonne raison de le faire.,We had a good reason for doing that.,We have a good reason for doing this.,We have a good reason for doing this.,We have a good reason for doing it.,We have a good reason for doing it.
6,Est-ce du bœuf ou du porc ?,Is this beef or pork?,Is the bread or the porc?,Is the bread or the porc?,Is the bread or the porc?,Is boiled or cooked pork?
7,Choisis celui que tu préfères.,Pick the one you prefer.,What are your preferences?,What are your preferences?,What are your preferences?,What are your preferences?
